# Streaming com Apache Kafka (demo complementar)

Este notebook demonstra o **mesmo cenario de negocio** do streaming da pipeline principal --
eventos de "atualizacao de indicador de alfabetizacao" quase em tempo real -- so que usando
**Apache Kafka** (broker local, do jeito como foi ensinado em algumas aulas) em vez do Amazon Kinesis
usado na pipeline de producao na AWS.

**Por que criamos esse notebook separado?** A pipeline principal (`notebooks/pipeline_alfabetizacao.ipynb`)
roda 100% na AWS e ja foi validada contra a conta real, usando Kinesis por ser um servico
gerenciado (sem broker para manter no ar). Este notebook mostra o mesmo padrao de streaming
implementado com Kafka -- a tecnologia vista em aula -- rodando localmente no Colab, sem
depender de infraestrutura AWS. Ver `README.md`, secao "Decisoes arquiteturais", para a
comparacao completa Kafka vs. Kinesis.

**So roda no Google Colab** (ou outro ambiente Linux/Debian com `apt-get`) -- o setup do broker
usa comandos de sistema que nao existem no Windows

## 1. Setup do Kafka (broker local)

Sobe ZooKeeper + Kafka Broker em background, como na Parte 1 da aula. Leva ~1-2 minutos na
primeira vez (baixa o Kafka); nas proximas execucoes reaproveita o que ja foi baixado.

In [1]:
import subprocess, os, time

KAFKA_VERSION = "3.7.0"
SCALA_VERSION = "2.13"
KAFKA_DIR     = f"/opt/kafka_{SCALA_VERSION}-{KAFKA_VERSION}"
KAFKA_URL     = f"https://archive.apache.org/dist/kafka/{KAFKA_VERSION}/kafka_{SCALA_VERSION}-{KAFKA_VERSION}.tgz"
KAFKA_ARCHIVE = "/tmp/kafka.tgz"
BIN           = f"{KAFKA_DIR}/bin"
BOOTSTRAP     = "localhost:9092"
TOPIC         = "alfabetizacao-eventos"

if subprocess.run(["java", "-version"], capture_output=True).returncode != 0:
    os.system("apt-get install -y -q default-jre-headless 2>/dev/null")
if subprocess.run(["which", "nc"], capture_output=True).returncode != 0:
    os.system("apt-get install -y -q netcat-openbsd 2>/dev/null")

if not os.path.isfile(f"{BIN}/zookeeper-server-start.sh"):
    print(f"Baixando Kafka {KAFKA_VERSION}...")
    os.system(f"wget -q {KAFKA_URL} -O {KAFKA_ARCHIVE}")
    os.system(f"tar -xzf {KAFKA_ARCHIVE} -C /opt/")
print("Kafka disponivel em", KAFKA_DIR)

Baixando Kafka 3.7.0...
Kafka disponivel em /opt/kafka_2.13-3.7.0


In [2]:
# Sobe ZooKeeper e aguarda a porta 2181
os.system("pkill -f zookeeper 2>/dev/null; sleep 1")
zk_proc = subprocess.Popen(
    f"{BIN}/zookeeper-server-start.sh {KAFKA_DIR}/config/zookeeper.properties".split(),
    stdout=open("/tmp/zookeeper.log", "w"), stderr=subprocess.STDOUT, preexec_fn=os.setsid,
)

def porta_aberta(porta):
    return subprocess.run(["nc", "-z", "-w1", "localhost", str(porta)], capture_output=True).returncode == 0

for _ in range(20):
    if porta_aberta(2181):
        print("ZooKeeper rodando na porta 2181")
        break
    time.sleep(1)
else:
    raise RuntimeError("ZooKeeper nao respondeu -- veja /tmp/zookeeper.log")

ZooKeeper rodando na porta 2181


In [3]:
# Sobe o Broker Kafka e aguarda a porta 9092
kafka_proc = subprocess.Popen(
    f"{BIN}/kafka-server-start.sh {KAFKA_DIR}/config/server.properties".split(),
    stdout=open("/tmp/kafka.log", "w"), stderr=subprocess.STDOUT,
)
time.sleep(8)

for _ in range(20):
    if porta_aberta(9092):
        print("Kafka Broker rodando na porta 9092")
        break
    time.sleep(1)
else:
    raise RuntimeError("Kafka nao respondeu -- veja /tmp/kafka.log")

Kafka Broker rodando na porta 9092


## 2. Criar o topico e instalar o cliente Python

In [4]:
!pip install kafka-python-ng --quiet

import json, uuid, random
from datetime import datetime, timezone
from kafka import KafkaProducer, KafkaConsumer, KafkaAdminClient
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError

admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP)
try:
    admin.create_topics([NewTopic(name=TOPIC, num_partitions=3, replication_factor=1)])
    print(f"Topico '{TOPIC}' criado (3 particoes)")
except TopicAlreadyExistsError:
    print(f"Topico '{TOPIC}' ja existe")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.8/232.8 kB 9.6 MB/s eta 0:00:00
Topico 'alfabetizacao-eventos' criado (3 particoes)


## 3. Producer -- os mesmos eventos sinteticos da pipeline AWS

Mesmo formato de evento usado em `src/bronze/streaming_producer.py` (o producer real, que envia
para o Kinesis) -- aqui trocamos so o destino, para deixar clara a equivalencia entre as duas
implementacoes. A **key** de particionamento e o `id_municipio`, garantindo que eventos do mesmo
municipio sempre vao para a mesma particao (ordem preservada).

In [5]:
# Mesma amostra de municipios do producer real (src/bronze/streaming_producer.py)
SAMPLE_MUNICIPIOS = [
    ("3550308", "SP", "Sao Paulo"),
    ("3304557", "RJ", "Rio de Janeiro"),
    ("2927408", "BA", "Salvador"),
    ("2304400", "CE", "Fortaleza"),
    ("1302603", "AM", "Manaus"),
    ("4106902", "PR", "Curitiba"),
]

def gerar_evento():
    id_municipio, sigla_uf, nome = random.choice(SAMPLE_MUNICIPIOS)
    return {
        "event_id": str(uuid.uuid4()),
        "event_type": "atualizacao_indicador_alfabetizacao",
        "id_municipio": id_municipio,
        "sigla_uf": sigla_uf,
        "nome_municipio": nome,
        "ano": datetime.now(timezone.utc).year,
        "percentual_alfabetizado": round(random.uniform(55.0, 95.0), 2),
        "amostra_avaliada": random.randint(50, 5000),
        "ingested_at": datetime.now(timezone.utc).isoformat(),
    }

producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: k.encode("utf-8"),
    acks="all",
    retries=3,
)

print(f"Enviando 20 eventos para o topico '{TOPIC}'...\n")
for _ in range(20):
    evento = gerar_evento()
    meta = producer.send(TOPIC, key=evento["id_municipio"], value=evento).get(timeout=5)
    print(f"  {evento['nome_municipio']:<16} ({evento['sigla_uf']}) -> particao {meta.partition}, offset {meta.offset}")
producer.flush()

Enviando 20 eventos para o topico 'alfabetizacao-eventos'...

  Salvador         (BA) -> particao 0, offset 0
  Curitiba         (PR) -> particao 2, offset 0
  Sao Paulo        (SP) -> particao 1, offset 0
  Fortaleza        (CE) -> particao 2, offset 1
  Fortaleza        (CE) -> particao 2, offset 2
  Curitiba         (PR) -> particao 2, offset 3
  Salvador         (BA) -> particao 0, offset 1
  Sao Paulo        (SP) -> particao 1, offset 1
  Salvador         (BA) -> particao 0, offset 2
  Salvador         (BA) -> particao 0, offset 3
  Rio de Janeiro   (RJ) -> particao 2, offset 4
  Salvador         (BA) -> particao 0, offset 4
  Sao Paulo        (SP) -> particao 1, offset 2
  Curitiba         (PR) -> particao 2, offset 5
  Curitiba         (PR) -> particao 2, offset 6
  Manaus           (AM) -> particao 2, offset 7
  Curitiba         (PR) -> particao 2, offset 8
  Rio de Janeiro   (RJ) -> particao 2, offset 9
  Sao Paulo        (SP) -> particao 1, offset 3
  Sao Paulo        (SP) ->

## 4. Consumer -- lendo e agregando os eventos

Consome o topico e calcula um resumo por municipio (media do percentual, quantidade de
eventos) -- equivalente, em espirito, ao que a Lambda `streaming-consumer` faz ao gravar em
`bronze/streaming_indicador/` na pipeline AWS.

In [6]:
from collections import defaultdict

consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=BOOTSTRAP,
    group_id="grupo-alfabetizacao-demo",
    auto_offset_reset="earliest",
    value_deserializer=lambda b: json.loads(b.decode("utf-8")),
    key_deserializer=lambda b: b.decode("utf-8") if b else None,
    consumer_timeout_ms=5000,
)

por_municipio = defaultdict(list)
total = 0

for msg in consumer:
    evento = msg.value
    por_municipio[evento["nome_municipio"]].append(evento["percentual_alfabetizado"])
    total += 1

consumer.close()

print(f"Total de eventos consumidos: {total}\n")
print(f"{'Municipio':<18} {'Eventos':>8} {'% medio':>10}")
print("-" * 38)
for municipio, valores in sorted(por_municipio.items()):
    print(f"{municipio:<18} {len(valores):>8} {sum(valores)/len(valores):>9.2f}%")

ERROR:kafka.consumer.fetcher:Fetch to node 0 failed: Cancelled: <BrokerConnection node_id=0 host=d2c5454ed818:9092 <connected> [IPv4 ('172.28.0.12', 9092)]>


Total de eventos consumidos: 20

Municipio           Eventos    % medio
--------------------------------------
Curitiba                  5     77.58%
Fortaleza                 2     76.02%
Manaus                    1     73.91%
Rio de Janeiro            2     70.98%
Salvador                  5     80.60%
Sao Paulo                 5     79.27%


## 5. Kafka vs. Kinesis -- por que a pipeline de producao usa Kinesis

| | Kafka (este notebook) | Amazon Kinesis (pipeline AWS) |
|---|---|---|
| Infraestrutura | Broker que voce sobe e mantem (aqui, local no Colab) | Servico gerenciado, sem servidor para administrar |
| Uso neste projeto | Demonstracao do padrao ensinado em aula | Ingestao streaming real da pipeline em producao |
| Escala em produção | Precisaria de Amazon MSK (Kafka gerenciado) na AWS | Ja é nativo da AWS, integra direto com Lambda |
| Custo em repouso | Cluster ficaria ligado o tempo todo (ou precisa subir/derrubar) | On-demand: paga só pelo que é escrito/lido |

Os dois resolvem o mesmo problema (ingestão de eventos quase em tempo real, particionados por
chave). A pipeline principal deste projeto usa Kinesis por já estar na AWS e não exigir um
cluster para manter no ar; este notebook existe para demonstrar o domínio de Kafka, a
tecnologia usada nas aulas de streaming.